### Génération de polaires - Regression

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# 1. Charger et nettoyer les données
df = pd.read_parquet("../Cleaned_Data/Cleaned_merge.parquet")
df["TWA"] = df["TWA"] % 360
df["TWA"] = np.where(
    df["TWA"] > 180,
    360 - df["TWA"],
    df["TWA"]
)
df['TWS'] = np.where(df['TWS'] > 100, np.nan, df['TWS'])
df_clean = df[["TWS", "TWA", "BSP"]].dropna()
# Ajout de points d'ancrage : BSP = 0 quand TWA = 0
tws_unique = df_clean["TWS"].unique()
anchor_points = pd.DataFrame({
    "TWS": tws_unique,
    "TWA": 0.0,
    "BSP": 0.0
})
df_clean = pd.concat([df_clean, anchor_points], ignore_index=True)

# 2. Préparer les données
X = df_clean[["TWS", "TWA"]].to_numpy()
y = df_clean["BSP"].to_numpy()

# 3. Régression polynomiale (degré 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

model = LinearRegression()
model.fit(X_poly, y)

# Prédictions sur l'ensemble d'entraînement
y_pred = model.predict(X_poly)

# Métriques de performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)

print("="*50)
print("PERFORMANCE DU MODÈLE")
print("="*50)
print(f"R² Score:           {r2:.4f} (0-1, 1 = parfait)")
print(f"Mean Squared Error: {mse:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f} nœuds")
print(f"Mean Absolute Error: {mae:.4f} nœuds")
print(f"Nombre de points:   {len(y)}")
print("="*50)

# 4. Créer la polaire (grille de prédictions)
tws_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 20)
twa_range = np.linspace(0, 180, 20)
tws_grid, twa_grid = np.meshgrid(tws_range, twa_range)

# Prédictions
predictions = []
for tws, twa in zip(tws_grid.flatten(), twa_grid.flatten()):
    sog_pred = model.predict(poly.transform([[tws, twa]]))
    predictions.append(sog_pred[0])

sog_grid = np.array(predictions).reshape(tws_grid.shape)

# 5. Utiliser la polaire
def get_bsp(tws, twa):
    # Appliquer la symétrie
    twa_norm = twa % 360
    if twa_norm > 180:
        twa_norm = 360 - twa_norm
    return model.predict(poly.transform([[tws, twa_norm]]))[0]

print(f"BSP à TWS=12, TWA=90: {get_bsp(12, 90):.2f}")

# 6. Visualiser polaire symétrique + résidus
fig = plt.figure(figsize=(16, 10))

# Subplot 1 : Polaire
ax1 = fig.add_subplot(121, projection='polar')

# Tracer des courbes pour différents TWS
colors = plt.cm.viridis(np.linspace(0, 1, len(tws_range)))
for i, tws_val in enumerate(tws_range[::2]):  # tous les 2 points
    bsp_for_tws = []
    # Tracer de 0° à 360° avec symétrie
    twa_full = np.linspace(0, 360, 180)
    for twa_val in twa_full:
        bsp = get_bsp(tws_val, twa_val)
        bsp_for_tws.append(max(0, bsp))  # éviter valeurs négatives
    
    ax1.plot(np.deg2rad(twa_full), bsp_for_tws, label=f'TWS={tws_val:.1f}', 
            color=colors[i*2], linewidth=2)

# Configuration de l'affichage polaire
ax1.set_theta_offset(np.pi/2)  # 0° en haut
ax1.set_theta_direction(-1)     # sens anti-horaire
ax1.set_ylim(0, y.max() * 1.2)
ax1.set_xlabel('TWA (°)')
ax1.legend(loc='upper left', bbox_to_anchor=(1.1, 1.1))
ax1.set_title('Polaire Symétrique - BSP vs TWA/TWS', pad=20)
ax1.grid(True)

# Subplot 2 : Prédictions vs Réalité
ax2 = fig.add_subplot(222)
ax2.scatter(y, y_pred, alpha=0.5, s=20)
ax2.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
ax2.set_xlabel('BSP Réel (nœuds)')
ax2.set_ylabel('BSP Prédit (nœuds)')
ax2.set_title(f'Prédictions vs Réalité\n(R²={r2:.4f})')
ax2.grid(True, alpha=0.3)

# Subplot 3 : Résidus
ax3 = fig.add_subplot(224)
residus = y - y_pred
ax3.scatter(y_pred, residus, alpha=0.5, s=20)
ax3.axhline(y=0, color='r', linestyle='--', lw=2)
ax3.set_xlabel('BSP Prédit (nœuds)')
ax3.set_ylabel('Résidus (nœuds)')
ax3.set_title(f'Résidus (RMSE={rmse:.4f})')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 1. Diviser les données : 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Créer et entraîner le Random Forest avec 150 arbres
rf_model = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 3. Prédictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# 4. Métriques
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_test_pred)

# 5. Affichage
print("="*60)
print("RANDOM FOREST - 150 ARBRES")
print("="*60)
print(f"Données d'entraînement: {len(X_train)} points (80%)")
print(f"Données de test: {len(X_test)} points (20%)")
print("-"*60)
print(f"R² Score (Train):       {r2_train:.4f}")
print(f"R² Score (Test):        {r2_test:.4f}")
print(f"Mean Squared Error:     {mse_test:.4f}")
print(f"Root Mean Squared Error: {rmse_test:.4f} nœuds")
print(f"Mean Absolute Error:    {mae_test:.4f} nœuds")
print("="*60)

# 6. Feature importance
print("\nImportance des features:")
print(f"  TWS: {rf_model.feature_importances_[0]:.4f}")
print(f"  TWA: {rf_model.feature_importances_[1]:.4f}")


In [ ]:
# Génération de la polaire avec Random Forest
fig = plt.figure(figsize=(16, 10))

# Subplot 1 : Polaire
ax1 = fig.add_subplot(121, projection='polar')

# Tracer des courbes pour différents TWS
colors = plt.cm.viridis(np.linspace(0, 1, len(tws_range)))
for i, tws_val in enumerate(tws_range[::2]):  # tous les 2 points
    bsp_for_tws = []
    # Tracer de 0° à 360° avec symétrie
    twa_full = np.linspace(0, 360, 180)
    for twa_val in twa_full:
        # Appliquer symétrie comme dans le premier modèle
        twa_norm = twa_val % 360
        if twa_norm > 180:
            twa_norm = 360 - twa_norm
        bsp = rf_model.predict([[tws_val, twa_norm]])[0]
        bsp_for_tws.append(max(0, bsp))  # éviter valeurs négatives
    
    ax1.plot(np.deg2rad(twa_full), bsp_for_tws, label=f'TWS={tws_val:.1f}', 
            color=colors[i*2], linewidth=2)

# Configuration de l'affichage polaire
ax1.set_theta_offset(np.pi/2)  # 0° en haut
ax1.set_theta_direction(-1)     # sens anti-horaire
ax1.set_ylim(0, y.max() * 1.2)
ax1.set_xlabel('TWA (°)')
ax1.legend(loc='upper left', bbox_to_anchor=(1.1, 1.1))
ax1.set_title('Polaire Random Forest (150 arbres) - BSP vs TWA/TWS', pad=20)
ax1.grid(True)

# Subplot 2 : Prédictions vs Réalité (Test)
ax2 = fig.add_subplot(222)
ax2.scatter(y_test, y_test_pred, alpha=0.5, s=20, color='orange')
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax2.set_xlabel('BSP Réel (nœuds)')
ax2.set_ylabel('BSP Prédit (nœuds)')
ax2.set_title(f'RF Prédictions vs Réalité (Test)\n(R²={r2_test:.4f})')
ax2.grid(True, alpha=0.3)

# Subplot 3 : Résidus
ax3 = fig.add_subplot(224)
residus_rf = y_test - y_test_pred
ax3.scatter(y_test_pred, residus_rf, alpha=0.5, s=20, color='orange')
ax3.axhline(y=0, color='r', linestyle='--', lw=2)
ax3.set_xlabel('BSP Prédit (nœuds)')
ax3.set_ylabel('Résidus (nœuds)')
ax3.set_title(f'Résidus RF (RMSE={rmse_test:.4f})')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# 1. Normaliser les données pour le réseau de neurones
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Construire le réseau de neurones [12, 6, 4]
nn_model = MLPRegressor(
    hidden_layer_sizes=(12, 6, 4),
    activation='tanh',
    solver='adam',
    max_iter=25000,
    random_state=42,
    verbose=False
)

# 3. Entraîner le modèle
nn_model.fit(X_train_scaled, y_train)

# 4. Évaluer sur les données de test
y_train_pred_nn = nn_model.predict(X_train_scaled)
y_test_pred_nn = nn_model.predict(X_test_scaled)

r2_train_nn = r2_score(y_train, y_train_pred_nn)
r2_test_nn = r2_score(y_test, y_test_pred_nn)
mse_test_nn = mean_squared_error(y_test, y_test_pred_nn)
rmse_test_nn = np.sqrt(mse_test_nn)
mae_test_nn = mean_absolute_error(y_test, y_test_pred_nn)

# 6. Affichage
print("="*60)
print("NEURAL NETWORK [12, 6, 4] - TANH - ADAM - 25000 EPOCHS")
print("="*60)
print(f"Données d'entraînement: {len(X_train)} points (80%)")
print(f"Données de test: {len(X_test)} points (20%)")
print("-"*60)
print(f"R² Score (Train):       {r2_train_nn:.4f}")
print(f"R² Score (Test):        {r2_test_nn:.4f}")
print(f"Mean Squared Error:     {mse_test_nn:.4f}")
print(f"Root Mean Squared Error: {rmse_test_nn:.4f} nœuds")
print(f"Mean Absolute Error:    {mae_test_nn:.4f} nœuds")
print("="*60)

# 7. Courbe d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(nn_model.loss_curve_, label='Train Loss (MSE)', linewidth=1)
axes[0].set_xlabel('Itérations')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Courbe de perte - Neural Network')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].axhline(mae_test_nn, color='tab:orange', linestyle='--', linewidth=2, label=f'MAE finale (test) = {mae_test_nn:.4f} nœuds')
axes[1].text(0.5, 0.5, 'MAE par itération non disponible\navec sklearn MLPRegressor', transform=axes[1].transAxes, ha='center', va='center', fontsize=10, color='gray', style='italic')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (nœuds)')
axes[1].set_title('Courbe MAE - Neural Network')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 8. Polaire Neural Network
import warnings
warnings.filterwarnings('ignore')

fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(121, projection='polar')
ax2 = fig.add_subplot(122)

tws_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 20)
colors = plt.cm.viridis(np.linspace(0, 1, len(tws_range)))

for i, tws_val in enumerate(tws_range[::2]):
    bsp_for_tws = []
    twa_full = np.linspace(0, 360, 180)
    for twa_val in twa_full:
        twa_norm = twa_val % 360
        if twa_norm > 180:
            twa_norm = 360 - twa_norm
        bsp = nn_model.predict(scaler.transform([[tws_val, twa_norm]]))[0]
        bsp_for_tws.append(max(0, bsp))
    ax1.plot(np.deg2rad(twa_full), bsp_for_tws, label=f'TWS={tws_val:.1f}',
             color=colors[i*2], linewidth=1.5)

# Configuration de l'affichage polaire
ax1.set_theta_offset(np.pi/2)
ax1.set_theta_direction(-1)
ax1.set_ylim(0, y.max() * 1.2)
ax1.set_xlabel('TWA (°)')
ax1.legend(loc='upper left', bbox_to_anchor=(1.1, 1.1))
ax1.set_title('Polaire Neural Network - BSP vs TWA/TWS', pad=20)
ax1.grid(True)

ax2.scatter(y_test, y_test_pred_nn, alpha=0.5, s=20, color='green')
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax2.set_xlabel('BSP Réel (nœuds)')
ax2.set_ylabel('BSP Prédit (nœuds)')
ax2.set_title(f'NN Prédictions vs Réalité (Test)\\n(R²={r2_test_nn:.4f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
